In [1]:
!pip install mne

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import numpy as np
import pandas as pd
from pathlib import Path
import mne
from scipy.signal import welch

ROOT = "/content/drive/MyDrive/Dissertation/Dataset"
INTERIM = Path(ROOT) / "data" / "interim"
OUTDIR  = Path(ROOT) / "data" / "features" / "video_windowed"
OUTDIR.mkdir(parents=True, exist_ok=True)

SUBJECTS = [f"s{i:02d}" for i in range(1, 33)]
print("INTERIM:", INTERIM)
print("OUTDIR :", OUTDIR)
print("Subjects:", len(SUBJECTS))



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 38.5 MB/s eta 0:00:00
Mounted at /content/drive
INTERIM: /content/drive/MyDrive/Dissertation/Dataset/data/interim
OUTDIR : /content/drive/MyDrive/Dissertation/Dataset/data/features/video_windowed
Subjects: 32


In [2]:
LABEL_DIR = Path(ROOT) / "data" / "features" / "labels"
THR_AVG = 5.5
THR_TAG = str(THR_AVG).replace(".", "p")

def load_subject_labels(subj: str) -> np.ndarray:
    p = LABEL_DIR / f"{subj}_labels_thr{THR_TAG}_avgVA.npz"
    if not p.exists():
        raise FileNotFoundError(f"Missing labels file: {p}")
    z = np.load(p, allow_pickle=True)
    y = z["y"].astype(int)
    assert y.shape == (40,), f"{subj}: expected (40,) labels, got {y.shape}"
    return y

print("s01 label balance:", np.unique(load_subject_labels("s01"), return_counts=True))


s01 label balance: (array([0, 1]), array([22, 18]))


In [3]:
EEG_CANONICAL = [
    'Fp1','AF3','F3','F7','FC5','FC1','C3','T7','CP5','CP1','P3','P7','PO3','O1',
    'Oz','Pz','Fp2','AF4','Fz','F4','F8','FC6','FC2','Cz','C4','T8','CP6','CP2',
    'P4','P8','PO4','O2'
]

def load_subject_epochs(subj: str):
    ep_path = INTERIM / f"{subj}-epo60.fif"
    if not ep_path.exists():
        raise FileNotFoundError(f"Missing epochs: {ep_path}")

    epochs = mne.read_epochs(str(ep_path), preload=True, verbose=False)

    # Pick + reorder to canonical 32 EEG channels
    missing = [ch for ch in EEG_CANONICAL if ch not in epochs.ch_names]
    if missing:
        raise RuntimeError(f"{subj}: Missing EEG channels: {missing}")

    epochs = epochs.copy().pick(EEG_CANONICAL)  # this enforces order

    X = epochs.get_data().astype(np.float32)  # (40, 32, T)
    assert X.shape[0] == 40, f"{subj}: expected 40 trials, got {X.shape[0]}"
    return X, float(epochs.info["sfreq"]), epochs.ch_names


X_all, y_all, groups, vid_ids = [], [], [], []
sfreq_ref, ch_names_ref = None, None

for subj in SUBJECTS:
    Xs, sf, ch_names = load_subject_epochs(subj)
    ys = load_subject_labels(subj)

    if sfreq_ref is None:
        sfreq_ref = sf
        ch_names_ref = ch_names
    else:
        assert abs(sf - sfreq_ref) < 1e-6, f"sfreq mismatch at {subj}"
        assert ch_names == ch_names_ref, f"channel order mismatch at {subj} (fix Notebook 01)"

    X_all.append(Xs)
    y_all.append(ys)
    groups.append(np.array([subj]*40, dtype=object))
    vid_ids.append(np.arange(40))

X_all = np.concatenate(X_all, axis=0)      # (1280, 32, T)
y_all = np.concatenate(y_all, axis=0)      # (1280,)
groups = np.concatenate(groups, axis=0)    # (1280,)
vid_ids = np.concatenate(vid_ids, axis=0)  # (1280,)

print("X_all:", X_all.shape, "y_all:", y_all.shape, "sfreq:", sfreq_ref)
print("Balance:", np.unique(y_all, return_counts=True))
print("Channels:", len(ch_names_ref), ch_names_ref[:10])


/tmp/ipython-input-1777458839.py:12: RuntimeWarning: This filename (/content/drive/MyDrive/Dissertation/Dataset/data/interim/s01-epo60.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(str(ep_path), preload=True, verbose=False)
/tmp/ipython-input-1777458839.py:12: RuntimeWarning: This filename (/content/drive/MyDrive/Dissertation/Dataset/data/interim/s02-epo60.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(str(ep_path), preload=True, verbose=False)
/tmp/ipython-input-1777458839.py:12: RuntimeWarning: This filename (/content/drive/MyDrive/Dissertation/Dataset/data/interim/s03-epo60.fif) does not conform to MNE naming conventions. All epochs files should end with -epo.fif, -epo.fif.gz, _epo.fif or _epo.fif.gz
  epochs = mne.read_epochs(str(ep_path), preload=True, verbose=False

X_all: (1280, 32, 15361) y_all: (1280,) sfreq: 256.0
Balance: (array([0, 1]), array([731, 549]))
Channels: 32 ['Fp1', 'AF3', 'F3', 'F7', 'FC5', 'FC1', 'C3', 'T7', 'CP5', 'CP1']


In [4]:
# Windowing params
WIN_SEC = 8.0
STEP_SEC = 8.0   # non-overlap for speed (change to 4.0 if you want overlap)
FMIN, FMAX = 1.0, 40.0

BANDS = {
    "theta": (4, 8),
    "alpha": (8, 13),
    "beta":  (13, 30),
    "gamma": (30, 40),
}

ASYM_PAIRS = [
    ("Fp1", "Fp2"),
    ("F3", "F4"),
    ("F7", "F8"),
    ("FC5", "FC6"),
    ("T7", "T8"),
    ("P7", "P8"),
    ("O1", "O2"),
]

band_names = list(BANDS.keys())

def bandpower_from_psd(freqs, psd, f1, f2):
    idx = (freqs >= f1) & (freqs < f2)
    if not np.any(idx):
        return np.zeros((psd.shape[0],), dtype=np.float32)
    bp = np.trapz(psd[:, idx], freqs[idx], axis=1)
    return bp.astype(np.float32)

def extract_video_features(X_video, sfreq, ch_names):
    """
    X_video: (32, T) float32
    returns: 1D feature vector
    """
    n_ch, n_t = X_video.shape
    win_samp  = int(round(WIN_SEC * sfreq))
    step_samp = int(round(STEP_SEC * sfreq))
    starts = np.arange(0, n_t - win_samp + 1, step_samp)
    if len(starts) < 1:
        raise ValueError("No windows. Decrease WIN_SEC or check T.")

    # Welch params: fast and stable
    nperseg = min(256, win_samp)
    noverlap = nperseg // 2

    # Store log-bandpower per window
    bp_win = np.zeros((len(starts), n_ch, len(band_names)), dtype=np.float32)

    for wi, s0 in enumerate(starts):
        seg = X_video[:, s0:s0+win_samp]  # (32, win_samp)
        freqs, psd = welch(seg, fs=sfreq, axis=1, nperseg=nperseg, noverlap=noverlap, scaling="density")
        keep = (freqs >= FMIN) & (freqs <= FMAX)
        freqs2 = freqs[keep]
        psd2 = psd[:, keep]

        for bi, bn in enumerate(band_names):
            f1, f2 = BANDS[bn]
            bp = bandpower_from_psd(freqs2, psd2, f1, f2)
            bp_win[wi, :, bi] = np.log(bp + 1e-12)

    # Dynamics over windows: mean/std
    bp_mean = bp_win.mean(axis=0)  # (32, bands)
    bp_std  = bp_win.std(axis=0)   # (32, bands)

    feat = np.concatenate([bp_mean.reshape(-1), bp_std.reshape(-1)]).astype(np.float32)

    # Asymmetry features: alpha & beta (mean and std)
    name_to_idx = {n: i for i, n in enumerate(ch_names)}
    asym = []
    a = band_names.index("alpha")
    b = band_names.index("beta")
    for L, R in ASYM_PAIRS:
        if (L in name_to_idx) and (R in name_to_idx):
            li, ri = name_to_idx[L], name_to_idx[R]
            asym += [
                bp_mean[li, a] - bp_mean[ri, a],
                bp_std[li, a]  - bp_std[ri, a],
                bp_mean[li, b] - bp_mean[ri, b],
                bp_std[li, b]  - bp_std[ri, b],
            ]
    feat = np.concatenate([feat, np.array(asym, dtype=np.float32)], axis=0)
    return feat

# Test feature dim
f0 = extract_video_features(X_all[0], sfreq_ref, ch_names_ref)
print("Feature dim D =", f0.shape[0])


Feature dim D = 284


/tmp/ipython-input-652965920.py:29: DeprecationWarning: `trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.
  bp = np.trapz(psd[:, idx], freqs[idx], axis=1)


In [5]:
CACHE = OUTDIR / f"X_video_win{WIN_SEC:g}_step{STEP_SEC:g}_bp_dyn_asym_thr{THR_TAG}.npz"

if CACHE.exists():
    z = np.load(CACHE, allow_pickle=True)
    X_feat = z["X_feat"].astype(np.float32)
    y = z["y"].astype(int)
    groups_saved = z["groups"].astype(object)
    vid_ids_saved = z["vid_ids"].astype(int)
    meta = dict(z["meta"].item())
    print("Loaded:", CACHE)
else:
    D = f0.shape[0]
    X_feat = np.zeros((X_all.shape[0], D), dtype=np.float32)
    for i in range(X_all.shape[0]):
        X_feat[i] = extract_video_features(X_all[i], sfreq_ref, ch_names_ref)
        if (i+1) % 100 == 0:
            print(f"Extracted {i+1}/{X_all.shape[0]}")

    y = y_all.copy()

    meta = dict(
        win_sec=WIN_SEC, step_sec=STEP_SEC, fmin=FMIN, fmax=FMAX,
        bands=BANDS, asym_pairs=ASYM_PAIRS,
        sfreq=sfreq_ref, channels=ch_names_ref,
        label_thr=THR_AVG
    )

    np.savez_compressed(
        CACHE,
        X_feat=X_feat,
        y=y,
        groups=groups,
        vid_ids=vid_ids,
        meta=meta
    )
    print("Saved:", CACHE)

print("X_feat:", X_feat.shape, "y:", y.shape)
print("Balance:", np.unique(y, return_counts=True))
print("Example row:", X_feat[0, :10])


Loaded: /content/drive/MyDrive/Dissertation/Dataset/data/features/video_windowed/X_video_win8_step8_bp_dyn_asym_thr5p5.npz
X_feat: (1280, 284) y: (1280,)
Balance: (array([0, 1]), array([731, 549]))
Example row: [-26.412052 -26.201818 -25.995718 -27.311832 -26.432964 -26.25935
 -25.968435 -27.331493 -26.48842  -26.253347]


In [6]:
FINAL_DATASET = OUTDIR / f"DEAP_video_windowed_features_thr{THR_TAG}.npz"
np.savez_compressed(
    FINAL_DATASET,
    X=X_feat.astype(np.float32),
    y=y.astype(np.int64),
    groups=groups,
    vid_ids=vid_ids,
    channels=np.array(ch_names_ref, dtype=object),
    feature_dim=np.array([X_feat.shape[1]]),
    meta=meta
)
print("Wrote final dataset:", FINAL_DATASET)


Wrote final dataset: /content/drive/MyDrive/Dissertation/Dataset/data/features/video_windowed/DEAP_video_windowed_features_thr5p5.npz


In [7]:
# No NaNs
assert np.isfinite(X_feat).all(), "Found NaNs/Infs in features"

# Check group distribution
print("Unique subjects in groups:", len(np.unique(groups)))

# Check that each subject has 40 videos
sub_counts = pd.Series(groups).value_counts().sort_index()
print(sub_counts.head())
assert (sub_counts.values == 40).all(), "Some subject does not have 40 trials"
print("Sanity checks passed.")


Unique subjects in groups: 32
s01    40
s02    40
s03    40
s04    40
s05    40
Name: count, dtype: int64
Sanity checks passed.
